# Week 54, Day 3: Training, validation and test sets

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w054/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Train, validate, test

Split the 120 trials at random into 80 for training and 40 for validation. For each lambda, fit ridge on the training trials and record the validation R². Refit on all 120 trials with the chosen lambda and compute the test R² once. Then repeat the split 50 times and record which lambda each split chooses.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
r = np.random.default_rng(543)
P = 55
w_true = np.zeros(P); w_true[:5] = [1.0, 0.8, -0.6, 0.5, -0.4]
X = r.normal(size=(120, P)); y = X @ w_true + r.normal(size=120)
X_te = r.normal(size=(5000, P)); y_te = X_te @ w_true + r.normal(size=5000)
A, A_te = np.c_[X, np.ones(120)], np.c_[X_te, np.ones(5000)]

def ridge_fit(A, y, lam):
    D = np.eye(A.shape[1]); D[-1, -1] = 0
    return np.linalg.solve(A.T @ A + lam * D, A.T @ y)

def r2(y, yhat):
    return 1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2)

lams = np.logspace(-2, 3, 26)
perm = r.permutation(120); tr_idx, va_idx = perm[:80], perm[80:]
val_r2 = ...
lam_val = lams[np.argmax(val_r2)]
honest_r2 = r2(y_te, A_te @ ridge_fit(A, y, lam_val))
best_possible = max(r2(y_te, A_te @ ridge_fit(A, y, lam)) for lam in lams)
lam_choices = []
for rep in range(50):
    p_ = r.permutation(120); t_, v_ = p_[:80], p_[80:]
    lam_choices.append(...)
print(f"chosen lambda {lam_val:.2f}: honest test R² {honest_r2:.3f}; best possible on the grid {best_possible:.3f}")
print(f"lambda chosen across 50 splits: median {np.median(lam_choices):.1f}, range {min(lam_choices):.2f} to {max(lam_choices):.0f}")

In [ ]:
import numpy as np
def test_split():
    "the training and validation trials are disjoint and together make all 120"
    assert len(set(tr_idx) & set(va_idx)) == 0 and len(tr_idx) + len(va_idx) == 120
def test_choice():
    "lam_val maximises validation R² over the grid"
    assert lam_val == lams[np.argmax(val_r2)]
def honest_r2():
    "the honest test R² is within 0.1 of the best achievable on the grid"
    assert honest_r2 > best_possible - 0.1 and honest_r2 <= best_possible + 1e-9
def test_spread():
    "lam_choices holds the lambda chosen on 50 different random splits"
    assert len(lam_choices) == 50 and len(set(lam_choices)) > 1

test_split()
test_choice()
test_spread()
print("All checks passed")

### Exercise: Selection on the test set

For each K, run 2,000 repetitions: draw the test-set accuracies of K random classifiers on 200 trials (each a binomial count divided by 200) and keep the largest. Store the mean in best_acc. Then check one selected model on a fresh 200-trial test set: its accuracy is just another binomial draw.

In [ ]:
Ks = [1, 2, 5, 10, 20, 50, 100, 200]
n_test, reps = 200, 2000
best_acc = np.zeros(len(Ks))
for k, K in enumerate(Ks):
    accs = r.binomial(n_test, 0.5, size=(reps, K)) / n_test
    best_acc[k] = ...
fresh_acc = np.mean(r.binomial(n_test, 0.5, size=reps) / n_test)
for K, b in zip(Ks, best_acc):
    print(f"best of {K:3d}: reported accuracy {b:.3f}")
print(f"the same selected models on a fresh test set: {fresh_acc:.3f}")

In [ ]:
import numpy as np
def test_sim():
    "best_acc holds the mean best test accuracy for each K, starting near 0.5 for K = 1"
    assert best_acc.shape == (len(Ks),) and abs(best_acc[0] - 0.5) < 0.01
def test_rise():
    "the reported accuracy rises with K and exceeds 0.56 for K = 100"
    assert np.all(np.diff(best_acc) > 0) and best_acc[Ks.index(100)] > 0.56
def test_fresh():
    "on a fresh test set, the selected model is back at chance"
    assert abs(fresh_acc - 0.5) < 0.02

test_sim()
test_rise()
test_fresh()
print("All checks passed")

### Exercise: Random against blocked splits

Write knn_predict(X_tr, y_tr, X_te), one-nearest-neighbour regression with Euclidean distance. Evaluate it with a random 80/20 split of the 600 trials and with a blocked split that holds out the last 20% of the session.

In [ ]:
T = 600
t = np.arange(T)
drift_x = np.cumsum(r.normal(size=(T, 3)), axis=0) / 10      # slow drifts in three neurons, unrelated to y
X_neural = drift_x + 0.03 * r.normal(size=(T, 3))
y_beh = np.cumsum(r.normal(size=T)) / 10 + 0.1 * r.normal(size=T)   # a slowly drifting behaviour

def knn_predict(X_tr, y_tr, X_te):
    d = ((X_te[:, None, :] - X_tr[None, :, :]) ** 2).sum(-1)
    return ...

perm = r.permutation(T); tr_r, te_r = perm[:480], perm[480:]
r2_random = r2(y_beh[te_r], knn_predict(X_neural[tr_r], y_beh[tr_r], X_neural[te_r]))
tr_b, te_b = t[:480], t[480:]
r2_blocked = ...
print(f"random split: R² {r2_random:.3f}")
print(f"blocked split: R² {r2_blocked:.3f}")

In [ ]:
import numpy as np
def test_knn():
    "knn_predict predicts each test point by its nearest training point's target"
    Xa = np.array([[0.0], [1.0], [5.0]]); ya = np.array([1.0, 2.0, 3.0])
    assert np.allclose(knn_predict(Xa, ya, np.array([[0.9], [4.0]])), [2.0, 3.0])
def test_leak():
    "the random split reports R² above 0.5; the blocked split reports R² below 0.1"
    assert r2_random > 0.5 and r2_blocked < 0.1

test_knn()
test_leak()
print("All checks passed")